In [3]:

import httpx
import time
import re
import unicodedata
from urllib.parse import urljoin, urlencode, urlparse
from urllib import robotparser
from bs4 import BeautifulSoup

In [9]:

UA = "StudyScraper(educational use)"
def fetch_html(url: str) ->str | None:
    try:
        r = httpx.get(url, headers={"User-Agent": UA}, timeout=15.0,  follow_redirects=True)
        if r.status_code == 200 and "text/html" in r.headers.get("content-type",""):
            return r.text
        return None
    except Exception as e:
        print("fetch error:", e)
        return None

In [ ]:
def is_junk_href(href: str) -> bool:
    if href is None:
        return True
    href = href.strip()
    if href == "" or href.startswith("#"):
        return True

    lower_href = href.lower()
    if lower_href.startswith("javascript:") or lower_href.startswith("mailto:") or lower_href.startswith("tel:"):
        return True

    if lower_href.endswith((".jpg", ".jpeg", ".png", ".gif", ".svg", ".pdf", ".zip")):
        return True
    return False

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

def extract_links(base_url: str, html: str) -> dict:
    
    soup = BeautifulSoup(html, "html.parser")
    seen: set[str] = set()
    results: dict[str, str] = {}
    
    base_domain = urljoin(base_url).netloc.lower()
    for a in soup.find_all("a"):
        href = a.get("href")
        # if is_junk_href(href):
        #     continue
        
        abs_url = urljoin(base_url, href)
        parsed = urlparse(abs_url)
        
        if parsed.scheme not in("http", "https"):
            continue
        # Optionally, skip external domains to focus on the same site:
        if parsed.netloc.lower() != base_domain:
            continue
        if abs_url in seen:
            continue
        
        seen.add(abs_url)
        page_html = fetch_html(abs_url)
        if not page_html:
            continue
        page_soup = BeautifulSoup(page_html, "html.parser")
        title_tag = page_soup.title
        title_text = title_tag.get_text(strip=True) if title_tag else ""
        results[title_text] = abs_url
    return results
    

TypeError: extract_links() missing 1 required positional argument: 'html'

In [ ]:

# html = fetch_html("https://www.ynet.co.il/home/0,7340,L-8,00.html")
# print(html)
# links = extract_links("https://www.ynet.co.il", html)
# print(len(links), "raw links")
# print(links)



<!DOCTYPE html>
<!--date generated - 2025-11-04T12:37:14.746611Z-->
<html lang='he' >
<head>
    <meta charset="UTF-8">
    <meta http-equiv="Pragma" content="no-cache"/>
    <title>ynet - חדשות, כלכלה, ספורט ובריאות - דיווחים שוטפים מהארץ ומהעולם</title>
        <meta name="description" content="אתר החדשות המוביל בישראל מבית ידיעות אחרונות. סיקור מלא של חדשות מישראל והעולם, ספורט, כלכלה, תרבות, אוכל, מדע וטבע, כל מה שקורה וכל מה שמעניין ב ynet"/><link rel='canonical' href='https://www.ynet.co.il/home/0,7340,L-8,00.html'>
        <meta itemprop="name" content="ynet - חדשות, כלכלה, ספורט ובריאות - דיווחים שוטפים מהארץ ומהעולם">
        
        <meta itemprop="description" content="אתר החדשות המוביל בישראל מבית ידיעות אחרונות. סיקור מלא של חדשות מישראל והעולם, ספורט, כלכלה, תרבות, אוכל, מדע וטבע, כל מה שקורה וכל מה שמעניין ב ynet">
                <meta property="og:type" content="website"/>
                <meta property="og:url" content="https://www.ynet.co.il/home/0,7340,L-8,00.html

# skipping the class of sites for now

fixing pharse or relitive links 


In [13]:
base_url = "https://apnews.com/"
JUNK_PREFIXES = ("mailto:", "javascript:", "tel:", "#")

def is_junk_href(href: str | None) -> bool:
    if not href:
        return True
    href = href.strip()
    return not href or href.startswith(JUNK_PREFIXES)


